# SCD Type 2 with Spark Declarative Pipelines Demo

This notebook demonstrates how to implement **Slowly Changing Dimension Type 2 (SCD Type 2)** using Spark Declarative Pipelines.

## What is SCD Type 2?
SCD Type 2 tracks historical changes by creating new records for each change while preserving old versions. This allows you to maintain a complete history of changes over time.

## Pipeline Architecture:
1. **Bronze Layer**: Raw data ingestion using Auto Loader from CSV files
2. **Silver Layer**: Cleaned data with SCD Type 2 history tracking

## Key Components:
- `dlt.table()`: Defines streaming tables
- `dlt.apply_changes()`: Applies CDC logic with SCD Type 2
- **Auto Loader**: Incrementally ingests new files from cloud storage

## Expected Output Schema:
The silver table will include:
- All original columns from source
- `__START_AT`: Timestamp when this version became active
- `__END_AT`: Timestamp when this version became inactive (NULL for current)
- `__CURRENT`: Boolean flag indicating if this is the current version

In [0]:
import dlt
from pyspark.sql.functions import col

## Prerequisites for Running This Demo

1. **Data Source**: Ensure CSV files exist in the volume path:
   - `/Volumes/workshop/scd2_demo/customer_volume/`
   - Files should include columns: `customer_id`, `last_updated`, and any customer attributes

2. **Expected CSV Format**:
   ```
   customer_id,name,email,city,last_updated
   CUST001,John Doe,john@email.com,NYC,2024-01-01 10:00:00
   ```

3. **Pipeline Setup**:
   - This notebook must be run as a **Spark Declarative Pipeline**, not a regular notebook
   - Create a pipeline in the UI and configure it to use this notebook
   - Configure target schema/catalog in pipeline settings

4. **To Test Changes**: Add new CSV files with updated customer records to trigger SCD Type 2 versioning

In [0]:
## Bronze Layer

In [0]:
# Source path where CSV files are stored
SOURCE_PATH = "/Volumes/workshop/scd2_demo/customer_volume/"

# Define bronze table using Auto Loader for incremental ingestion
@dlt.table(
    name="bronze_customers",
    comment="Raw customer records loaded using Auto Loader"
)
def bronze_customers():
    # Auto Loader automatically detects and processes new files
    # cloudFiles format enables incremental, fault-tolerant file processing
    return (
        spark.readStream
        .format("cloudFiles")  # Auto Loader streaming source
        .option("cloudFiles.format", "csv")  # Source file format
        .option("header", "true")  # First row contains column names
        .option("inferSchema", "true")  # Automatically detect column types
        .load(SOURCE_PATH)
    )

In [0]:
# Create the target streaming table for SCD Type 2
# This table will be populated by dlt.apply_changes() below
dlt.create_streaming_table(
    name="silver_customers",
    comment="Customer dimension table with SCD Type 2 history tracking"
)

In [0]:
# Apply CDC (Change Data Capture) logic with SCD Type 2
# This automatically handles inserts, updates, and history tracking
dlt.apply_changes(
    target="silver_customers",  # Destination table
    source="bronze_customers",  # Source table with changes
    keys=["customer_id"],  # Primary key to identify unique records
    sequence_by=col("last_updated"),  # Column to order changes chronologically
    stored_as_scd_type=2  # Enable SCD Type 2 history (creates __START_AT, __END_AT, __CURRENT)
)

## Understanding the apply_changes() Configuration

**Parameters explained:**
- `target`: The destination streaming table (silver_customers)
- `source`: The source table with change data (bronze_customers)
- `keys`: Column(s) that uniquely identify a record (customer_id)
- `sequence_by`: Column used to order changes (last_updated timestamp)
- `stored_as_scd_type=2`: Enables SCD Type 2 history tracking

**How it works:**
When a customer record changes:
1. The old record's `__END_AT` is set to the new record's timestamp
2. The old record's `__CURRENT` is set to False
3. A new record is inserted with `__CURRENT=True` and `__END_AT=NULL`
4. Both versions are preserved in the table

## How to Query SCD Type 2 Tables

```sql
-- Get current (active) records only
SELECT * FROM silver_customers WHERE __CURRENT = true

-- Get all historical versions for a specific customer
SELECT * 
FROM silver_customers 
WHERE customer_id = 'CUST123'
ORDER BY __START_AT

-- Get records active at a specific point in time
SELECT * 
FROM silver_customers
WHERE __START_AT <= '2024-01-01' 
  AND (__END_AT > '2024-01-01' OR __END_AT IS NULL)

-- Count versions per customer
SELECT customer_id, COUNT(*) as version_count
FROM silver_customers
GROUP BY customer_id
ORDER BY version_count DESC
```

## Best Practices for SCD Type 2 Pipelines

### Key Considerations:

1. **Choose the Right Sequence Column**
   - Use a monotonically increasing timestamp or sequence number
   - Common choices: `last_updated`, `modified_timestamp`, `version_number`
   - Must be present in every record

2. **Primary Key Selection**
   - Keys should uniquely identify a business entity
   - Composite keys are supported: `keys=["customer_id", "account_id"]`
   - Ensure keys are never NULL

3. **Performance Tips**
   - Use Delta Lake optimizations (OPTIMIZE, Z-ORDER)
   - Consider partitioning large tables by date
   - Monitor checkpoint locations for streaming sources

4. **Common Troubleshooting**
   - **Missing sequence column**: Ensure `sequence_by` column exists in source
   - **Duplicate processing**: Check Auto Loader checkpoint location
   - **NULL keys**: Validate that key columns are never NULL
   - **Wrong order**: Verify sequence column has correct chronological values

### Demo Testing Workflow:

1. **Initial Load**: Add first batch of customer CSV files
2. **Run Pipeline**: Execute to create initial records
3. **Simulate Changes**: Add new CSV with updated customer info (same customer_id, newer timestamp)
4. **Run Pipeline Again**: Watch SCD Type 2 magic happen!
5. **Query Results**: Use the SQL examples above to see versioned history